# Download the TinyStories dataset

In [ ]:
!mkdir -p data
!cd data

!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
!wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!cd ..


# Import Packages/Modules

In [ ]:
import os
import regex as re
from typing import BinaryIO
from itertools import repeat
import multiprocessing as mp
from collections import defaultdict, Counter
from concurrent.futures import ProcessPoolExecutor
from typing import Iterator

# Finding Chunk Boundaries
- Forked from `assignment1-basics/cs336_basics/pretokenization_example.py`.

In [2]:

def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

# Pre-Tokenization Function

In [ ]:
pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def pre_tokenization(
                    file_path: str, 
                    begin: int, 
                    terminate: int
                ) -> dict[tuple[bytes, ...], int]:
    
    with open(file_path, 'rb') as f:
        f.seek(begin)
        chunk = f.read(terminate - begin).decode("utf-8", errors="ignore")
        count: dict[tuple[bytes, ...], int] = {}

        special_tokens = ["<|endoftext|>"]
        special_pattern = "|".join([re.escape(token) for token in special_tokens])
        chunk = re.split(special_pattern, chunk)

        for chunk_split in chunk:
            for m in re.finditer(pattern, chunk_split):
                bytes_tuple = tuple(bytes([ch]) for ch in m.group(0).encode('utf-8'))
                count[bytes_tuple] = count.get(bytes_tuple, 0) + 1
    return count

# Multiprocessing ProcessPoolExecutor Function
- Distributes tasks to child processes and collects, aggregates the data.

In [ ]:
def mp_regex(
            file_path: str, 
            start: list[int] , 
            end: list[int], 
            num_workers: int | None = os.cpu_count()
        ) -> dict[tuple[bytes, ...], int]:

    ctx = mp.get_context("fork")
    
    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        results: Iterator[dict[tuple[bytes, ...], int]] = executor.map(pre_tokenization, repeat(file_path), start, end)
        
    pre_token_counts: Counter[tuple[bytes, ...]] = Counter()
    for worker_dict in results:
        pre_token_counts.update(worker_dict)
    return dict(pre_token_counts)

# Example code block
- Intended for quick testing of code, algorithms.

In [26]:
example_text = """
low low <|startoftext|> low low low
lower lower widest <|endoftext|> widest widest
newest newest newest <|im_start|> newest newest newest
"""

special_tokens = ["<|startoftext|>", "<|endoftext|>", "<|im_start|>"]

pattern = "|".join([re.escape(token) for token in special_tokens])
print(pattern)
example_text = re.split(pattern, example_text)
print(example_text)

pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

example_count: dict[tuple[bytes, ...], int] = {}

for string in example_text:
    for m in re.finditer(pattern, string):
        bytes_tuple = tuple(bytes([ch]) for ch in m.group(0).encode('utf-8'))
        example_count[bytes_tuple] = example_count.get(bytes_tuple, 0) + 1

print(example_count)

for token in special_tokens:
    print(token.encode('utf-8'))

<\|startoftext\|>|<\|endoftext\|>|<\|im_start\|>
['\nlow low ', ' low low low\nlower lower widest ', ' widest widest\nnewest newest newest ', ' newest newest newest\n']
{(b'\n',): 4, (b'l', b'o', b'w'): 1, (b' ', b'l', b'o', b'w'): 4, (b' ',): 3, (b'l', b'o', b'w', b'e', b'r'): 1, (b' ', b'l', b'o', b'w', b'e', b'r'): 1, (b' ', b'w', b'i', b'd', b'e', b's', b't'): 3, (b'n', b'e', b'w', b'e', b's', b't'): 1, (b' ', b'n', b'e', b'w', b'e', b's', b't'): 5}
b'<|startoftext|>'
b'<|endoftext|>'
b'<|im_start|>'


In [ ]:
example_text = """
low low low low low
lower lower widest <|endoftext|> widest widest
newest newest newest newest newest newest
"""


pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

example_count: dict[tuple[bytes, ...], int] = {}

for m in re.finditer(pattern, example_text):
    bytes_tuple = tuple(ch.encode('utf-8') for ch in m.group(0))
    example_count[bytes_tuple] = example_count.get(bytes_tuple, 0) + 1

print(example_count)

# Parallelized Pre-Tokenization Execution

In [11]:
train_data = "./data/TinyStoriesV2-GPT4-train.txt"

with open(train_data, "rb") as f:
    num_processes = 4
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    start = boundaries[:-1]
    end = boundaries[1:]
    
    print(f"{start}\n{end}")
    
    pre_token_count = mp_regex(train_data, start, end, num_processes)
    
    print(len(pre_token_count))

# print(f"\n{pre_token_count}")

[0, 556938507, 1113876594, 1670815354]
[556938507, 1113876594, 1670815354, 2227753162]
59933


# Finding adjacent pairs and their appearance count function

In [12]:
def find_pairs(pre_token_count: dict[tuple[bytes, ...], int]) -> dict[tuple[bytes, bytes], int]:
    
    pairs: dict[tuple[bytes, bytes], int] = defaultdict(int)
    for tpl, appear in pre_token_count.items():
        for i in range(len(tpl)-1):
            pairs[(tpl[i], tpl[i+1])] += appear
    return pairs

# Merge Function

In [ ]:
def merge(
        pre_token_count: dict[tuple[bytes, ...], int], 
        pair: tuple[bytes, bytes], 
        verbose: bool = False
    ) -> dict[tuple[bytes, ...], int]:
    
    total_merges: int = 0
    new_token_count: dict[tuple[bytes, ...], int] = defaultdict(int)
    
    for tpl, appear in pre_token_count.items():
        i = 0
        end_flag: bool = False
        merge_happen: bool = False
        new_tpl: list[bytes] = []
        
        if len(tpl) < 2:
            new_token_count[tpl] = appear
            continue
            
        while i < len(tpl)-1:
            if (pair[0], pair[1]) == (tpl[i], tpl[i+1]):
                new_tpl.append(pair[0]+pair[1])
                end_flag = True if i == len(tpl)-2 else False
                merge_happen = True
                i += 2
                total_merges += 1
                continue
            new_tpl.append(tpl[i])
            i += 1
            
        if not end_flag:
            new_tpl.append(tpl[-1])
             
        new_token_count[tuple(new_tpl)] = appear
        if verbose and merge_happen:
            print(f"Merge Successful with {pair=} resulting in new string {tuple(new_tpl)=}")
    
    if verbose:
        print(f"Total merges: {total_merges}")
    return new_token_count

# Train Function
- Trains the tokenizer for a given number of steps.

In [16]:
def train(
        merge_iters: int, 
        pre_token_count: dict[tuple[bytes, ...], int]
    ) -> tuple[dict[int, bytes], dict[tuple[bytes, ...], int], list[tuple[bytes, bytes]]]:
    
    i2b_vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    
    merge_order: list[tuple[bytes, bytes]] = []
    
    for _ in range(merge_iters):
        pairs: dict[tuple[bytes, bytes], int] = find_pairs(pre_token_count)
        max_pair: tuple[bytes, bytes] = max(pairs, key=lambda k: (pairs[k], k))

        v_idx: int = max(i2b_vocab) + 1
        b_string: bytes = max_pair[0] + max_pair[1]

        merge_order.append(max_pair)

        i2b_vocab[v_idx] = b_string
        # b2i_vocab[b_string] = v_idx
        
        pre_token_count = merge(pre_token_count, max_pair)

    return i2b_vocab, pre_token_count, merge_order
        
    

In [17]:
i2b_vocab, pre_token_count, merge_order = train(merge_iters=10000-257, pre_token_count=pre_token_count)

# Vocab Inspection

In [20]:
print(f"{'Length of Vocab':<50}: {len(i2b_vocab)}\n")
# print("-"*54)
# for key, value in i2b_vocab.items():
#     print(f"{'Index':<10}: {key:>5} {'|':^5} {'Byte_String':<12}: {repr(value):>15}")



Length of Vocab                                   : 9999

